# Stable Diffusion, and a walk between two prompts

A pretrained text-to-image model, negative prompts, and the slerp interpolation that is the best visual argument in the book for what a neural network is.

**Runs on:** GPU with 12 GB+ · large download &nbsp;·&nbsp; **Slides:** [Chapter 17 — Image Generation](../../../course-web-slides/ch17/index.html) &nbsp;·&nbsp; **Section:** 03 — Text-to-image models

---

## Loading it

In [ ]:
import keras
import keras_hub

height, width = 512, 512
task = keras_hub.models.TextToImage.from_preset(
    "stable_diffusion_3_medium",
    image_shape=(height, width, 3),
    dtype="float16",
)
print(type(task).__name__)

Like `CausalLM` in chapter 16, `TextToImage` is a high-level task class wrapping tokenization and the whole diffusion loop into one `generate()` call. `dtype="float16"` halves the memory — chapter 18 explains what that costs.

## Generating

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

prompt = "A NASA astronaut riding an origami elephant in New York City"
image = task.generate(prompt)

plt.figure(figsize=(7, 7))
plt.imshow(image); plt.axis("off"); plt.title(prompt, fontsize=10)
plt.show()

Look closely and you will find artifacts — the book's example has an elephant with duplicated tusks. **Two causes, and only one is fixable.**

Drawing a human in a space suit on a paper elephant needs anatomy and physics the model lacks. But we are also using the **smallest** Stable Diffusion 3 release, about 3 billion parameters; the 9-billion version produces substantially fewer artifacts.

## Negative prompts

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 6.6))
a1.imshow(task.generate(prompt)); a1.set_title("prompt only"); a1.axis("off")
a2.imshow(task.generate({"prompts": prompt,
                         "negative_prompts": "blue color"}))
a2.set_title('negative prompt: "blue color"'); a2.axis("off")
plt.tight_layout(); plt.show()

There is nothing magic here. Train on **triplets** — `(image, positive_prompt, negative_prompt)` where the negative is words that do *not* describe the image — and the denoiser learns to move toward one and away from the other.

**A control surface built entirely out of how the training data was labelled.**

## Watching the denoising happen

In [ ]:
from PIL import Image

def display(images):
    return Image.fromarray(np.concatenate(images, axis=1))

steps = [5, 10, 15, 20, 25]
imgs = [task.generate(prompt, num_steps=s) for s in steps]

fig, axes = plt.subplots(1, len(steps), figsize=(18, 4))
for ax, im, s in zip(axes, imgs, steps):
    ax.imshow(im); ax.set_title(f"{s} steps"); ax.axis("off")
plt.tight_layout(); plt.show()

**The same weights throughout** — only the number of times they were applied changed. Exactly the parameter from notebook 04, exposed on a production model.

## Taking generate() apart

In [ ]:
from keras import random, ops

def get_text_embeddings(prompt):
    token_ids = task.preprocessor.generate_preprocess([prompt])
    negative_token_ids = task.preprocessor.generate_preprocess([""])
    return task.backbone.encode_text_step(token_ids, negative_token_ids)

def denoise_with_text_embeddings(embeddings, num_steps=28, guidance_scale=7.0):
    latents = random.normal((1, height // 8, width // 8, 16))
    for step in range(num_steps):
        latents = task.backbone.denoise_step(
            latents, embeddings, step, num_steps, guidance_scale)
    return task.backbone.decode_step(latents)[0]

def scale_output(x):
    x = ops.convert_to_numpy(x)
    x = np.clip((x + 1.0) / 2.0, 0.0, 1.0)
    return np.round(x * 255.0).astype("uint8")

embeddings = get_text_embeddings(prompt)
print("latent shape:", (1, height // 8, width // 8, 16))
print("Stable Diffusion denoises in a COMPRESSED latent space and")
print("decodes to pixels only at the end -- which is what makes")
print("512x512 affordable.")

## Four tensors, not one

In [ ]:
print([tuple(np.array(x).shape) for x in embeddings])
print()
print("The authors pass BOTH the final embedded vector AND the last")
print("representation of the whole token sequence, for both the positive")
print("and the negative prompt:")
print("  [0] positive prompt's encoder sequence  (1, 154, 4096)")
print("  [1] negative prompt's encoder sequence  (1, 154, 4096)")
print("  [2] positive prompt's encoder vector    (1, 2048)")
print("  [3] negative prompt's encoder vector    (1, 2048)")

## Spherical interpolation, and why not linear

In [ ]:
def slerp(t, v1, v2):
    v1, v2 = ops.cast(v1, "float32"), ops.cast(v2, "float32")
    v1_norm = ops.linalg.norm(ops.ravel(v1))
    v2_norm = ops.linalg.norm(ops.ravel(v2))
    dot = ops.sum(v1 * v2 / (v1_norm * v2_norm))
    theta_0 = ops.arccos(dot)
    sin_theta_0 = ops.sin(theta_0)
    theta_t = theta_0 * t
    sin_theta_t = ops.sin(theta_t)
    s0 = ops.sin(theta_0 - theta_t) / sin_theta_0
    s1 = sin_theta_t / sin_theta_0
    return s0 * v1 + s1 * v2

# Why it matters: linear interpolation shrinks the norm.
a = np.random.normal(size=2048).astype("float32")
b = np.random.normal(size=2048).astype("float32")
lin = [(1-t) * a + t * b for t in np.linspace(0, 1, 9)]
sph = [np.array(slerp(float(t), a, b)) for t in np.linspace(0, 1, 9)]

plt.figure(figsize=(6.5, 4))
plt.plot(np.linspace(0, 1, 9), [np.linalg.norm(v) for v in lin], "o-",
         label="linear")
plt.plot(np.linspace(0, 1, 9), [np.linalg.norm(v) for v in sph], "s-",
         label="spherical")
plt.xlabel("t"); plt.ylabel("vector norm"); plt.legend()
plt.title("Linear interpolation cuts through the inside of the sphere")
plt.show()

**The midpoint of a linear interpolation has a smaller norm than either endpoint** — it has left the surface. The text manifold is not actually spherical, but it is a smooth surface of vectors with roughly the same magnitude, and interpolating as if on a sphere is a much better approximation than as if on a line.

## The walk

In [ ]:
def interpolate_text_embeddings(e1, e2, start=0, stop=1, num=10):
    out = []
    for t in np.linspace(start, stop, num):
        out.append((slerp(float(t), e1[0], e2[0]),
                    e1[1],
                    slerp(float(t), e1[2], e2[2]),
                    e1[3]))
    return out

prompt1 = "A friendly dog looking up in a field of flowers"
prompt2 = ("A horrifying, tentacled creature hovering over a field of flowers")

e1 = get_text_embeddings(prompt1)
e2 = get_text_embeddings(prompt2)

images = []
for et in interpolate_text_embeddings(e1, e2, start=0.5, stop=0.6, num=9):
    images.append(scale_output(denoise_with_text_embeddings(et)))

fig, axes = plt.subplots(1, 9, figsize=(20, 2.6))
for ax, im in zip(axes, images):
    ax.imshow(im); ax.axis("off")
plt.suptitle("Nine images across one tenth of the path between two prompts",
             y=1.1)
plt.tight_layout(); plt.show()

The walk runs from **0.5 to 0.6** out of [0, 1] — zoomed into the middle, right where the morph becomes visually obvious.

> This might feel like magic the first time you try it, but there's nothing magic about it — **interpolation is fundamental to the way deep neural networks learn.**

Chapter 15 argued this algebraically about embedding spaces. Here it is, frame by frame. **Same claim, two proofs** — and it is the note the book chooses to end its modelling on.

---

## What to take away

- Stable Diffusion denoises in a compressed latent space and decodes at the end.
- Negative prompts are a control surface built from how the training data was labelled.
- Slerp stays on the manifold; linear interpolation cuts through the inside and loses meaning.
- **Deep networks are interpolation machines**, and this walk is the clearest picture of it in the course.